# 🌶️ Spice · สะพานยืมการ์ดจอ

โน้ตบุ๊กนี้จะเปลี่ยนเครื่อง Colab ของคุณให้กลายเป็น **เครื่องประมวลผลของเว็บแอป Spice**

สิ่งที่จะเกิดขึ้นเมื่อกดรัน:
1. ตรวจสอบว่าได้การ์ดจอ (Tesla T4) มาจริง
2. จับคู่กับบัญชี Google ของคุณด้วย **รหัสจับคู่** ที่ได้จากหน้าเว็บ
3. เมานต์ Google Drive ด้วย `rclone` โดยใช้สิทธิ์ที่คุณอนุญาตไว้แล้วในเว็บ
4. เฝ้ารอคิวงาน แล้วรันโมเดลบน GPU ส่งผลกลับหน้าเว็บแบบสด

---
### ⚠️ ก่อนเริ่ม — ต้องตั้งค่า GPU ก่อน
เมนู **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**

ถ้าไม่ตั้ง เครื่องจะรันบน CPU ซึ่งช้ามากจนใช้งานจริงไม่ได้


## ขั้นที่ 1 · ตรวจการ์ดจอที่ได้มา


In [ ]:
!nvidia-smi || echo '❌ ยังไม่ได้เปิด GPU — ไปที่ Runtime → Change runtime type → T4 GPU'


## ขั้นที่ 2 · ใส่ค่าการเชื่อมต่อ

เอาค่าทั้งสองมาจากหน้าเว็บ Spice → **เครื่อง GPU → + เชื่อมเครื่องใหม่**

| ช่อง | เอามาจากไหน |
|---|---|
| `SERVER` | URL ของเว็บ Spice ของคุณ เช่น `https://spice.example.com` |
| `PAIR_CODE` | รหัส 8 ตัวที่หน้าเว็บสร้างให้ เช่น `K7QD-2M9X` (ใช้ได้ครั้งเดียว อายุ 15 นาที) |


In [ ]:
# @title ใส่ค่าแล้วกดรัน { display-mode: "form" }
SERVER    = "https://spice.example.com"  # @param {type:"string"}
PAIR_CODE = "XXXX-XXXX"                  # @param {type:"string"}
WORKER_NAME = ""                          # @param {type:"string"}
MOUNT_DRIVE = True                        # @param {type:"boolean"}

SERVER = SERVER.strip().rstrip('/')
PAIR_CODE = PAIR_CODE.strip().upper()
assert SERVER.startswith('http'), 'SERVER ต้องขึ้นต้นด้วย http:// หรือ https://'
assert len(PAIR_CODE) >= 8, 'ยังไม่ได้ใส่รหัสจับคู่จากหน้าเว็บ'
print(f'✅ จะเชื่อมไปที่ {SERVER} ด้วยรหัส {PAIR_CODE}')


## ขั้นที่ 3 · ติดตั้งสิ่งที่ต้องใช้

ใช้เวลาประมาณ 1–2 นาทีในครั้งแรก


In [ ]:
!pip -q install requests
!curl -sSL https://rclone.org/install.sh | sudo bash > /dev/null 2>&1 && echo '✅ ติดตั้ง rclone แล้ว' || echo '⚠️ ข้าม rclone (ยังใช้งานระบบได้ แต่จะเมานต์ Drive ไม่ได้)'


## ขั้นที่ 4 · เริ่มให้ยืมการ์ดจอ 🚀

เซลล์นี้จะ **รันค้างไว้** — อย่าปิดแท็บนี้

กลับไปที่หน้าเว็บได้เลย เครื่องจะขึ้นสถานะ “ออนไลน์” ภายในไม่กี่วินาที

หยุดให้ยืมเมื่อไหร่ก็ได้ด้วยการกดปุ่มหยุด (■) ที่เซลล์นี้


In [ ]:
import subprocess, sys

# ดึงตัวแทนเครื่องรุ่นล่าสุดจากเซิร์ฟเวอร์ของคุณเอง
subprocess.run(['curl','-sSL',f'{SERVER}/colab/bootstrap.py','-o','spice_agent.py'], check=True)

args = [sys.executable,'spice_agent.py','--server',SERVER,'--pair',PAIR_CODE]
if WORKER_NAME.strip():
    args += ['--name', WORKER_NAME.strip()]
if not MOUNT_DRIVE:
    args += ['--no-drive']

!{' '.join(args)}


---
## ❓ แก้ปัญหาที่พบบ่อย

**“รหัสจับคู่ไม่ถูกต้อง หมดอายุ หรือถูกใช้ไปแล้ว”**
→ รหัสใช้ได้ครั้งเดียวและมีอายุ 15 นาที กลับไปกดขอรหัสใหม่ในหน้าเว็บ

**เครื่องขึ้นออฟไลน์ทั้งที่เซลล์ยังรันอยู่**
→ ตรวจว่า `SERVER` เข้าถึงได้จากอินเทอร์เน็ต (ถ้ารันบนเครื่องตัวเองต้องเปิด tunnel เช่น Cloudflare Tunnel หรือ ngrok ก่อน)

**เมานต์ Drive ไม่สำเร็จ**
→ ไปที่หน้า *Drive & rclone* ในเว็บแล้วกด “เชื่อม Google Drive” ใหม่อีกครั้ง ระบบยังรันงานได้ตามปกติแม้เมานต์ไม่ได้

**CUDA out of memory**
→ เลือกโมเดลที่เล็กลง หรือดูที่หน้าเลือกโมเดลว่าอันไหนขึ้นว่า “รันได้เลย”

**Colab ตัดการเชื่อมต่อเอง**
→ เป็นข้อจำกัดของบัญชีฟรี งานที่ค้างจะถูกโยนกลับเข้าคิวอัตโนมัติ เปิดเซลล์นี้ใหม่แล้วจับคู่อีกครั้งได้เลย
